<a href="https://colab.research.google.com/github/bogdangradinaru/ai-safety/blob/main/notebooks/part-1/01-supervised-kitti-yolo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Supervised Learning for Automotive Object Detection

## KITTI Dataset and YOLO

This practical laboratory demonstrates how
supervised learning is applied to automotive perception.

## Learning Objectives

- Understand supervised learning.
- Explore automotive datasets.
- Train a YOLO model.
- Evaluate detection performance.

# Supervised Learning for Automotive Object Detection

## KITTI Dataset and YOLO

This practical laboratory demonstrates how supervised
deep learning can be applied to automotive perception.

We will train an object detection model to identify:
- Cars
- Pedestrians
- Cyclists
- Trucks

## Learning Objectives

By completing this laboratory, you will understand:

1. Supervised learning fundamentals.
2. Automotive image datasets and annotations.
3. YOLO object detection architecture.
4. Model training and validation.
5. Object detection performance metrics.
6. AI safety implications of incorrect predictions.

**Tools:** Python, PyTorch, Ultralytics YOLO, KITTI.

**Environment:** Google Colab with GPU recommended.

## 1. Environment Setup

We will use the Ultralytics framework for training and
evaluating the YOLO object detection model.

In [ ]:
%pip install -q ultralytics

In [ ]:
import os
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import torch

from ultralytics import YOLO
from ultralytics.data.utils import (
    check_det_dataset,
    img2label_paths
)

print("All libraries imported successfully.")

In [ ]:
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU.")

## 2. KITTI Automotive Dataset

KITTI contains real-world automotive images captured
around Karlsruhe, Germany.

The Ultralytics object detection configuration provides:

- 5,985 training images
- 1,496 validation images
- 8 object classes

Source:
https://www.cvlibs.net/datasets/kitti/

In [ ]:
dataset = check_det_dataset("kitti.yaml")

print("Training images:", dataset["train"])
print("Validation images:", dataset["val"])
print("Classes:", dataset["names"])
print("Number of classes:", dataset["nc"])

In [ ]:
class_names = dataset["names"]

for class_id, class_name in class_names.items():
    print(f"{class_id}: {class_name}")

In [ ]:
0: car
1: van
2: truck
3: pedestrian
4: person_sitting
5: cyclist
6: tram
7: misc

## 3. Understanding Ground-Truth Labels

YOLO uses five values for each bounding box:

class_id x_center y_center width height

The coordinates are normalized relative to image dimensions.

For example:

0 0.52 0.45 0.20 0.30

This represents an object of class 0 with a bounding
box positioned at the specified normalized coordinates.

In [ ]:
train_dir = Path(dataset["train"])

image_files = sorted(
    p for p in train_dir.rglob("*")
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)

print("Number of training images:", len(image_files))
print("First image:", image_files[0])

In [ ]:
sample_image = image_files[0]

label_path = Path(
    img2label_paths([str(sample_image)])[0]
)

image = cv2.imread(str(sample_image))
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

height, width = image.shape[:2]

with open(label_path) as f:
    labels = f.readlines()

for line in labels:
    class_id, xc, yc, bw, bh = map(
        float, line.split()
    )

    x1 = int((xc - bw / 2) * width)
    y1 = int((yc - bh / 2) * height)
    x2 = int((xc + bw / 2) * width)
    y2 = int((yc + bh / 2) * height)

    cv2.rectangle(
        image, (x1, y1), (x2, y2), (0, 255, 0), 2
    )

    name = class_names[int(class_id)]

    cv2.putText(
        image, name, (x1, max(15, y1 - 5)),
        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2
    )

plt.figure(figsize=(14, 8))
plt.imshow(image)
plt.title("KITTI Ground-Truth Annotations")
plt.axis("off")
plt.show()

In [ ]:
class_counts = Counter()

label_files = img2label_paths(
    [str(p) for p in image_files]
)

for label_file in label_files:
    if not os.path.exists(label_file):
        continue

    with open(label_file) as f:
        for line in f:
            if line.strip():
                class_id = int(line.split()[0])
                class_counts[class_id] += 1

counts = pd.Series({
    class_names[i]: class_counts[i]
    for i in class_names
})

counts.sort_values().plot(
    kind="barh",
    figsize=(9, 5)
)

plt.title("KITTI Object Class Distribution")
plt.xlabel("Number of annotated objects")
plt.show()

In [ ]:
baseline_model = YOLO("yolo26n.pt")

In [ ]:
results = baseline_model.predict(
    source=str(sample_image),
    conf=0.25
)

annotated = results[0].plot()

plt.figure(figsize=(14, 8))
plt.imshow(cv2.cvtColor(
    annotated, cv2.COLOR_BGR2RGB
))
plt.axis("off")
plt.title("Baseline YOLO Detection")
plt.show()

In [ ]:
TRAIN_CONFIG = {
    "data": "kitti.yaml",
    "epochs": 20,
    "imgsz": 640,
    "batch": 8,
    "device": 0 if torch.cuda.is_available() else "cpu",
    "seed": 42,
    "plots": True,
    "project": "runs/kitti",
    "name": "supervised-baseline"
}

In [ ]:
model = YOLO("yolo26n.pt")

train_results = model.train(**TRAIN_CONFIG)